# 7 — DQN jako wstęp przed PPO/Actor-Critic

Ten notebook jest **krótki** i ma dać intuicję:
- tablicowe Q-learning → DQN (sieć),
- replay buffer,
- target network,
- proste ablation study (co się psuje bez tych elementów).

To jest **run & interpret**: kod dostajesz gotowy, Twoim zadaniem są wnioski.

TODO: WSTAW RYSUNEK (DQN): replay buffer + target network.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from typing import Optional, Dict, List, Tuple

# ---------- Viz helpers (consistent with Ch04) ----------
def pretty_matrix_as_grid(v: np.ndarray, nrow: int, ncol: int, decimals: int = 2):
    grid = np.asarray(v, dtype=float).reshape(nrow, ncol)
    with np.printoptions(precision=decimals, suppress=True):
        print(grid)

def action_arrows(pi_det: np.ndarray, nrow: int, ncol: int, arrows: Dict[int, str]):
    out = []
    for r in range(nrow):
        row = []
        for c in range(ncol):
            s = r * ncol + c
            row.append(arrows.get(int(pi_det[s]), '?'))
        out.append(' '.join(row))
    print('\n'.join(out))

def moving_average(x, window: int = 200):
    x = np.asarray(x, dtype=float)
    if len(x) < window:
        return x
    w = np.ones(window) / window
    return np.convolve(x, w, mode="valid")

# ---------- Environment wrapper: sample transitions from P[s][a] ----------
class _Discrete:
    def __init__(self, n: int):
        self.n = int(n)

class PModelEnv:
    \"\"\"Minimal env wrapper around model P[s][a].\"\"\"
    def __init__(self, P, start_state: int = 0, seed: int = 0):
        self.P = P
        self.nS = len(P)
        s0 = next(iter(P))
        self.nA = len(P[s0])
        self.action_space = _Discrete(self.nA)
        self.observation_space = _Discrete(self.nS)
        self.start_state = int(start_state)
        self.rng = np.random.default_rng(seed)
        self.s = self.start_state

    def reset(self, seed: Optional[int] = None):
        if seed is not None:
            self.rng = np.random.default_rng(int(seed))
        self.s = self.start_state
        return int(self.s), {}

    def step(self, a: int):
        a = int(a)
        outcomes = self.P[self.s][a]  # [(p, s2, r, terminated), ...]
        ps = np.array([o[0] for o in outcomes], dtype=float)
        idx = int(self.rng.choice(len(outcomes), p=ps/ps.sum()))
        p, s2, r, terminated = outcomes[idx]
        self.s = int(s2)
        return int(s2), float(r), bool(terminated), False, {}

# ---------- FrozenLake model builder (no Gym) ----------
def build_frozenlake_P(desc, is_slippery: bool = False):
    \"\"\"FrozenLake in P[s][a] format. Actions: 0=L,1=D,2=R,3=U.\"\"\"
    desc = np.asarray([list(row) for row in desc], dtype="<U1")
    nrow, ncol = desc.shape
    nS, nA = nrow * ncol, 4

    LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3
    moves = {LEFT:(0,-1), DOWN:(1,0), RIGHT:(0,1), UP:(-1,0)}

    def to_s(r,c): return r*ncol+c
    def step_from(r,c,a):
        dr,dc = moves[a]
        r2,c2 = r+dr, c+dc
        if r2<0 or r2>=nrow or c2<0 or c2>=ncol:
            r2,c2 = r,c
        return r2,c2

    P = {s: {a: [] for a in range(nA)} for s in range(nS)}
    for r in range(nrow):
        for c in range(ncol):
            s = to_s(r,c)
            tile = desc[r,c]
            if tile in ("H","G"):
                for a in range(nA):
                    P[s][a] = [(1.0, s, 0.0, True)]
                continue

            for a in range(nA):
                if is_slippery:
                    candidates = [(a-1)%4, a, (a+1)%4]
                    probs = [1/3, 1/3, 1/3]
                else:
                    candidates = [a]
                    probs = [1.0]

                outcomes = []
                for a_real, p in zip(candidates, probs):
                    r2,c2 = step_from(r,c,a_real)
                    s2 = to_s(r2,c2)
                    tile2 = desc[r2,c2]
                    terminated = tile2 in ("H","G")
                    reward = 1.0 if tile2 == "G" else 0.0
                    outcomes.append((float(p), int(s2), float(reward), bool(terminated)))

                merged = {}
                for p, s2, rwd, term in outcomes:
                    key = (s2, rwd, term)
                    merged[key] = merged.get(key, 0.0) + p
                P[s][a] = [(p, s2, rwd, term) for (s2, rwd, term), p in merged.items()]
    return P, nS, nA, nrow, ncol, desc

# ---------- Shared env instances ----------
desc4 = ["SFFF","FHFH","FFFH","HFFG"]
P_fl_det, nS, nA, nrow, ncol, _ = build_frozenlake_P(desc4, is_slippery=False)
P_fl_slip, _, _, _, _, _ = build_frozenlake_P(desc4, is_slippery=True)

env = PModelEnv(P_fl_det, start_state=0, seed=0)
env_slip = PModelEnv(P_fl_slip, start_state=0, seed=0)

arrows_fl = {0:"←",1:"↓",2:"→",3:"↑"}

print("FrozenLake map:")
print("\\n".join(desc4))
print("n_states =", nS, "n_actions =", nA)


In [ ]:
# ---------- Common model-free helpers ----------
def epsilon_greedy_action(q_s: np.ndarray, eps: float, rng: np.random.Generator) -> int:
    if rng.random() < eps:
        return int(rng.integers(0, len(q_s)))
    return int(np.argmax(q_s))

def generate_episode_det_policy(env: PModelEnv, pi_det: np.ndarray,
                                max_steps: int = 200, seed: Optional[int] = None):
    s, _ = env.reset(seed=seed)
    episode = []
    for _ in range(max_steps):
        a = int(pi_det[s])
        s2, r, done, trunc, _ = env.step(a)
        episode.append((s, a, r))
        if done or trunc:
            break
        s = s2
    return episode

def generate_episode_eps_greedy(env: PModelEnv, Q: np.ndarray, eps: float,
                                rng: np.random.Generator, max_steps: int = 200):
    s, _ = env.reset()
    episode = []
    for _ in range(max_steps):
        a = epsilon_greedy_action(Q[s], eps, rng)
        s2, r, done, trunc, _ = env.step(a)
        episode.append((s, a, r))
        if done or trunc:
            break
        s = s2
    return episode


## Baseline: tablicowe Q-learning (dla porównania)

Cel: mieć punkt odniesienia dla DQN.


In [ ]:
def epsilon_greedy_action(q_s: np.ndarray, eps: float, rng: np.random.Generator) -> int:
    if rng.random() < eps:
        return int(rng.integers(0, len(q_s)))
    return int(np.argmax(q_s))

def q_learning(env: PModelEnv,
               alpha: float = 0.5, gamma: float = 0.99,
               eps: float = 0.2, episodes: int = 20_000,
               seed: int = 0, max_steps: int = 200):
    nS = env.observation_space.n
    nA = env.action_space.n
    Q = np.zeros((nS, nA), dtype=float)
    rng = np.random.default_rng(seed)
    success_curve = []

    for _ in range(episodes):
        s, _ = env.reset(seed=int(rng.integers(0, 10_000_000)))
        success = 0
        for _ in range(max_steps):
            a = epsilon_greedy_action(Q[s], eps, rng)
            s2, r, done, trunc, _ = env.step(a)
            if r > 0:
                success = 1
            target = float(r) if (done or trunc) else float(r) + gamma * float(np.max(Q[s2]))
            Q[s, a] += alpha * (target - Q[s, a])
            s = s2
            if done or trunc:
                break
        success_curve.append(success)
    return np.asarray(success_curve, dtype=int)

base = q_learning(env, episodes=25_000, eps=0.2, alpha=0.5, seed=0)
plt.figure()
plt.plot(moving_average(base, 500))
plt.title("Tabular Q-learning baseline (det)")
plt.xlabel("epizod (wygładzone 500)")
plt.ylabel("success rate")
plt.grid(True)
plt.show()


---

## DQN: implementacja + ablations

Poniżej mamy DQN z:
- replay buffer,
- target network,
- epsilon-greedy behavior.

Eksperymenty (zalecane):
1) DQN pełny (replay + target)
2) DQN bez target network
3) DQN bez replay (online updates)

Uwaga: to ma pokazać stabilność, nie wykręcać najlepszy wynik.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random

class DQN(nn.Module):
    def __init__(self, nS, nA):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(nS, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, nA),
        )
    def forward(self, x):
        return self.net(x)

def one_hot_state(s: int, nS: int) -> np.ndarray:
    x = np.zeros(nS, dtype=np.float32)
    x[int(s)] = 1.0
    return x

def dqn_train(env: PModelEnv, episodes: int = 2500, gamma: float = 0.99,
              lr: float = 1e-3, batch_size: int = 64,
              buffer_size: int = 50_000, min_buffer: int = 500,
              eps_start: float = 1.0, eps_end: float = 0.05, eps_decay: float = 0.995,
              target_update: int = 200, max_steps: int = 200, seed: int = 0,
              use_replay: bool = True, use_target: bool = True):
    rng = np.random.default_rng(seed)
    random.seed(seed)
    torch.manual_seed(seed)

    nS = env.observation_space.n
    nA = env.action_space.n

    q = DQN(nS, nA)
    q_tgt = DQN(nS, nA)
    q_tgt.load_state_dict(q.state_dict())
    q_tgt.eval()

    opt = optim.Adam(q.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    buffer = deque(maxlen=buffer_size)

    success_curve = []
    loss_curve = []
    eps = eps_start
    global_step = 0

    for ep in range(episodes):
        s, _ = env.reset(seed=int(rng.integers(0, 10_000_000)))
        ep_success = 0

        for _ in range(max_steps):
            if rng.random() < eps:
                a = int(rng.integers(0, nA))
            else:
                with torch.no_grad():
                    qs = q(torch.tensor(one_hot_state(s, nS)).unsqueeze(0))
                    a = int(torch.argmax(qs, dim=1).item())

            s2, r, done, trunc, _ = env.step(a)
            if r > 0:
                ep_success = 1

            transition = (s, a, r, s2, done or trunc)
            if use_replay:
                buffer.append(transition)
            else:
                buffer.clear()
                buffer.append(transition)

            s = s2
            global_step += 1

            if len(buffer) >= min_buffer:
                batch = random.sample(buffer, min(batch_size, len(buffer)))
                S = torch.tensor(np.stack([one_hot_state(b[0], nS) for b in batch]))
                A = torch.tensor([b[1] for b in batch], dtype=torch.long)
                R = torch.tensor([b[2] for b in batch], dtype=torch.float32)
                S2 = torch.tensor(np.stack([one_hot_state(b[3], nS) for b in batch]))
                D = torch.tensor([b[4] for b in batch], dtype=torch.float32)

                q_sa = q(S).gather(1, A.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    if use_target:
                        q_next = q_tgt(S2).max(dim=1).values
                    else:
                        q_next = q(S2).max(dim=1).values
                    target = R + (1.0 - D) * gamma * q_next

                loss = loss_fn(q_sa, target)
                opt.zero_grad()
                loss.backward()
                opt.step()

                loss_curve.append(float(loss.item()))

                if use_target and (global_step % target_update == 0):
                    q_tgt.load_state_dict(q.state_dict())

            if done or trunc:
                break

        success_curve.append(ep_success)
        eps = max(eps_end, eps * eps_decay)

    return np.asarray(success_curve, dtype=int), np.asarray(loss_curve, dtype=float)

# --- Run 3 configs ---
configs = [
    ("DQN full (replay+target)", True, True, 0),
    ("No target", True, False, 1),
    ("No replay (online)", False, True, 2),
]

curves = []
for name, use_replay, use_target, sd in configs:
    sc, lc = dqn_train(env, seed=sd, use_replay=use_replay, use_target=use_target)
    curves.append((name, sc, lc))

plt.figure()
for name, sc, _ in curves:
    plt.plot(moving_average(sc, 200), label=name)
plt.title("DQN preview: success rate (det)")
plt.xlabel("epizod (wygładzone 200)")
plt.ylabel("success rate")
plt.grid(True)
plt.legend()
plt.show()


## Pytania do wniosków (oddanie)

Napisz 8–12 zdań:
1) Co daje replay buffer? (w kontekście korelacji próbek i stabilności)  
2) Co daje target network? (w kontekście bootstrapu)  
3) Która ablation psuje learning najbardziej i dlaczego?  
4) Jak to się ma do PPO/Actor-Critic (on-policy, stabilność)?
